In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))

from analysis import load_run, normalize_run
from analysis.aggregations import grs_by_user
from analysis.plots import plot_grs_by_user
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

## Load run

In [ ]:
FOLDER = "03-05-26--22_29_41/mnist-loss_tolerance_aware-1-0.1-1-1.0-True-{a61b7629-1a83-407e-9541-d09efb3e5baf}.pkl"

data_dir = REPO_ROOT / "experiment" / "data" / "sample"
if FOLDER:
    data_dir = data_dir / FOLDER

run = load_run(data_dir)
run = normalize_run(run)
run

## Compute loss diff % per participant per round

`user_mad_avg` is the MAD-filtered average loss the participant achieved.  
`prev_avg_round_val_after_mad` is the global baseline that round was scored against.  
A positive diff means the participant's loss was *worse* than baseline → punishment candidate.

In [ ]:
TOLERANCE_PCT = 15.0  # must match loss_tolerance_pct * 100 in your experiment config

contrib = run.contributions.copy()
users   = run.rounds_users.copy()

contrib['loss_diff_pct'] = (
    (contrib['user_mad_avg'] - contrib['prev_avg_round_val_after_mad'])
    / contrib['prev_avg_round_val_after_mad'] * 100
)

# Join behavior (constant per user — take first occurrence)
user_meta = users.groupby('user_id')[['behavior', 'role']].first().reset_index()
contrib = contrib.merge(user_meta, on='user_id', how='left')

# Join per-round reward/punishment outcome
reward_col = users[['round', 'user_id', 'is_reward']].copy()
contrib = contrib.merge(reward_col, on=['round', 'user_id'], how='left')

contrib[['round', 'user_id', 'behavior', 'user_mad_avg', 'prev_avg_round_val_after_mad', 'loss_diff_pct', 'is_reward']].sort_values(['user_id', 'round'])

## Plot: loss diff % per participant per round

In [ ]:
BEHAVIOR_COLORS = {
    'good':      'steelblue',
    'bad':       'red',
    'malicious': 'red',
    'freerider': 'purple',
}

fig, ax = plt.subplots(figsize=(13, 5))

ax.axhline(0,             color='black',  linewidth=0.8, linestyle='--', label='baseline (0%)')
ax.axhline(TOLERANCE_PCT, color='orange', linewidth=1.2, linestyle=':',  label=f'+{TOLERANCE_PCT}% tolerance (ε)')

for user_id, grp in contrib.groupby('user_id'):
    behavior = grp['behavior'].iloc[0]
    color    = BEHAVIOR_COLORS.get(behavior, 'gray')
    grp_sorted = grp.sort_values('round')
    ax.plot(
        grp_sorted['round'],
        grp_sorted['loss_diff_pct'],
        marker='o', markersize=4,
        label=f'#{user_id} ({behavior})',
        color=color,
    )
    # Mark punishment rounds with an X
    punished = grp_sorted[grp_sorted['is_reward'] == False]
    ax.scatter(punished['round'], punished['loss_diff_pct'], marker='x', s=80, color=color, zorder=5)

ax.set_xlabel('Round')
ax.set_ylabel('Loss diff vs baseline (%)')
ax.set_title('Per-participant loss diff % per round  (✕ = punished that round)')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Per-participant summary statistics

In [ ]:
stats = (
    contrib
    .groupby(['user_id', 'behavior'])['loss_diff_pct']
    .agg(mean='mean', std='std', median='median', min='min', max='max')
    .round(2)
    .reset_index()
)

punished_counts = (
    contrib[contrib['is_reward'] == False]
    .groupby('user_id')
    .size()
    .rename('punished_rounds')
)
total_counts = contrib.groupby('user_id').size().rename('total_rounds')
rates = pd.concat([punished_counts, total_counts], axis=1).fillna(0)
rates['punishment_rate_%'] = (rates['punished_rounds'] / rates['total_rounds'] * 100).round(1)

summary = stats.merge(rates, on='user_id', how='left')
summary

## Rounds where honest participants were punished

These are the cases of interest: `behavior == 'good'` but the participant received a punishment that round.

In [ ]:
honest_punished = contrib[
    (contrib['behavior'] == 'good') &
    (contrib['is_reward'] == False)
][['round', 'user_id', 'loss_diff_pct', 'user_mad_avg', 'prev_avg_round_val_after_mad']].copy()

honest_punished['above_tolerance'] = honest_punished['loss_diff_pct'] > TOLERANCE_PCT

print(f"Honest participants punished: {len(honest_punished)} time(s) across all rounds")
print(f"  of which loss_diff > {TOLERANCE_PCT}% (outside ε): {honest_punished['above_tolerance'].sum()}")
print(f"  of which loss_diff ≤ {TOLERANCE_PCT}% (inside ε, punished for other reason): {(~honest_punished['above_tolerance']).sum()}")
honest_punished.sort_values(['user_id', 'round'])

## Distribution of loss diff % by behavior

Shows whether honest and malicious/freerider participants occupy clearly separated regions.  
If they overlap significantly, the tolerance threshold needs adjustment.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for behavior, grp in contrib.groupby('behavior'):
    vals = grp['loss_diff_pct'].dropna()
    # clip extreme outliers for readability
    clipped = vals.clip(-200, 200)
    color = BEHAVIOR_COLORS.get(behavior, 'gray')
    ax.hist(clipped, bins=20, alpha=0.6, label=behavior, color=color)

ax.axvline(0,             color='black',  linewidth=1,   linestyle='--', label='0% (baseline)')
ax.axvline(TOLERANCE_PCT, color='orange', linewidth=1.2, linestyle=':',  label=f'+{TOLERANCE_PCT}% ε')
ax.set_xlabel('Loss diff vs baseline (%, clipped at ±200)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of per-round loss diff % by behavior')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Boxplot: loss diff % by behavior

Cleaner view of the spread — useful for judging whether the ε boundary separates behaviors.

In [ ]:
behaviors = contrib['behavior'].unique()
data_by_behavior = [
    contrib[contrib['behavior'] == b]['loss_diff_pct'].dropna().clip(-300, 500).values
    for b in behaviors
]

fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(data_by_behavior, labels=behaviors, patch_artist=True)
for patch, behavior in zip(bp['boxes'], behaviors):
    patch.set_facecolor(BEHAVIOR_COLORS.get(behavior, 'gray'))
    patch.set_alpha(0.6)

ax.axhline(0,             color='black',  linewidth=0.8, linestyle='--')
ax.axhline(TOLERANCE_PCT, color='orange', linewidth=1.2, linestyle=':',  label=f'+{TOLERANCE_PCT}% ε')
ax.set_ylabel('Loss diff vs baseline (%)')
ax.set_title('Loss diff % distribution by behavior (clipped)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## GRS over rounds — does reputation recover for punished honest participants?

In [ ]:
data = run.rounds_users
agg  = grs_by_user(data)
fig  = plot_grs_by_user(agg)

## Runtime warnings (Shapley violations etc.)

In [ ]:
run.warnings